# A Scalable Architecture for Different Branch-and-Bound Instantiations

Objective and bounding functions

- Gradient sum and sum of positive gradient elements only, fast bound computation needs access
- 

In [47]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

In [48]:
from numba import njit, int64, float64
from numba.experimental import jitclass
from numba.typed import List
from numba import types
import numpy as np


# PAIR_TYPE = types.Tuple((float64, int64))
TUPLE_TYPE = types.Tuple((float64, int64))

heap_spec = [
    ('data', types.ListType(TUPLE_TYPE))
]

@jitclass(heap_spec)
class Heap:

    def __init__(self):
        self.data = List.empty_list(TUPLE_TYPE)

    def push(self, key, idx):
        self.data.append((key, idx))
        i = len(self.data) - 1
        while i > 0:
            parent = (i - 1) // 2
            if self.data[i][0] <= self.data[parent][0]:
                break
            self.data[i], self.data[parent] = self.data[parent], self.data[i]
            i = parent

    def pop(self):
        top = self.data[0]
        last = self.data.pop()
        if len(self.data) == 0:
            return top
        self.data[0] = last
        i = 0
        while True:
            left = 2 * i + 1
            right = 2 * i + 2
            largest = i
            if left < len(self.data) and self.data[left][0] > self.data[largest][0]:
                largest = left
            if right < len(self.data) and self.data[right][0] > self.data[largest][0]:
                largest = right
            if largest == i:
                break
            self.data[i], self.data[largest] = self.data[largest], self.data[i]
            i = largest
        return top
    
heap = Heap()
heap.push(-1.0, 0)
heap.pop()

(-1.0, 0)

In [81]:
import numpy as np

from numba.experimental import jitclass
from numba.types import int64, float64
from numba.typed import List
from numba import njit

from optikon import Propositionalization, compute_bounds, equal_width_propositionalization, full_propositionalization
from testdata import mvn_with_correlation

@jitclass
class SimpleTreeSearchNode:

    key: int64[:]
    critical: int64[:]
    remaining: int64[:]
    support: int64[:]

    def __init__(self, key, critical, remaining, support):
        self.key = key
        self.critical = critical
        self.remaining =remaining
        self.support = support

@njit
def make_simple_root(x, prop):
    l, u = compute_bounds(x)
    remaining = prop.nontrivial(l, u, np.arange(len(prop)))
    empty = np.empty(0, dtype=np.int64)
    return SimpleTreeSearchNode(empty, empty, remaining, np.arange(len(x)))

def node_to_string(node):
    return f'Node({node.key}, {node.value}, {node.bound})'

# @njit
# def dummy_obj(node, x, y):
#     return float64(len(node.key))

@jitclass
class KeyLength:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return float64(len(node.key))
    
key_length = KeyLength()
    
@jitclass
class KeyLengthBound:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return float(len(node.key) + len(node.remaining))
    
key_length_bound = KeyLengthBound()

# @njit
# def dummy_bnd(node, x, y):
#     return float64(len(node.key) + len(node.remaining))

# @njit
# def weighted_support(node, x, y):
#     return y[node.support].sum()

@jitclass
class WeightedSupport:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return y[node.support].sum()

weighted_support = WeightedSupport()

# @njit
# def weighted_support_bound_naive(node, x, y):
#     pos = (y > 0)
#     return y[node.support][pos].sum() 

@jitclass
class WeightedSupportBoundNaive:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        pos = (y > 0)
        return y[node.support[pos[node.support]]].sum()
    
weighted_support_bound_naive = WeightedSupportBoundNaive()

SimpleTreeSearchNodeType = SimpleTreeSearchNode.class_type.instance_type

@jitclass
class SearchSpec:

    x: float64[:, :]
    y: float64[:]
    prop: Propositionalization
    max_depth: int64

    def __init__(self, x, y, prop, max_depth=4):
        self.x = x
        self.y = y
        self.prop = prop
        self.max_depth = max_depth


@jitclass
class SearchResult:

    best: SimpleTreeSearchNode
    value: float64
    bound: float64
    nodes_created: int64
    edges_tested: int64

    def __init__(self, best, value, nodes_created, edges_tested):
        self.best = best
        self.value = value
        self.nodes_created = nodes_created
        self.edges_tested = edges_tested

    def __str__(self):
        return self.best.key, self.value


@njit
def refinement(node, search_spec):
    res = List()
    for p_idx in range(len(node.remaining)):
        p = node.remaining[p_idx]

        _key = np.empty(len(node.key) + 1, dtype=np.int64)
        _key[:-1] = node.key
        _key[-1] = p
        _sup = node.support[search_spec.prop.support(p, search_spec.x[node.support])]
        
        _crit = np.empty(len(node.critical) + p_idx, dtype=np.int64)
        _crit[:len(node.critical)] = node.critical
        _crit[len(node.critical):] = node.remaining[:p_idx]
        l, u = compute_bounds(search_spec.x[_sup])
        if len(search_spec.prop.trivial(l, u, _crit))>0:
            continue

        _remaining = search_spec.prop.nontrivial(l, u, node.remaining[p_idx+1:])
        res.append(SimpleTreeSearchNode(_key, _crit, _remaining, _sup))
    return res


@njit
def run(spec, obj, bnd):
    heap = Heap()
    nodes = List.empty_list(SimpleTreeSearchNodeType) #List.empty_list(CanonicalTreeSearchNode.class_type.instance_type)
    freelist = List.empty_list(int64)

    root = make_simple_root(spec.x, spec.prop)
    nodes.append(root)
    heap.push(bnd.compute(root, spec.x, spec.y), 0)

    best = root
    best_value = obj.compute(root, spec.x, spec.y)
    created = 1
    non_canonical = 0

    while len(heap.data) > 0:
        bound, idx = heap.pop()
        node = nodes[idx]
        freelist.append(idx)

        if bound <= best_value:
            continue
        if len(node.key) >= spec.max_depth:
            continue

        children = refinement(node, spec)
        # children = node.refinement(spec)
        created += len(children)
        non_canonical += len(node.remaining) - len(children)

        for child in children:
            child_value = obj.compute(child, spec.x, spec.y)
            child_bound = bnd.compute(child, spec.x, spec.y)
            if child_value > best_value:
                best = child
                best_value = child_value

            if len(freelist) > 0:
                reuse_idx = freelist.pop()
                nodes[reuse_idx] = child
                heap.push(child_bound, reuse_idx)
            else:
                nodes.append(child)
                heap.push(child_bound, len(nodes) - 1)

    return SearchResult(best, best_value, created, created-1+non_canonical)
    

x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)
prop = equal_width_propositionalization(x) # full_propositionalization(x) #
search = SearchSpec(x, y, prop, 8)
run(search, key_length, key_length_bound).best.key

array([ 8, 13, 23, 32, 38, 48, 53, 68])

In [82]:
%timeit run(search, key_length, key_length_bound).best.key

1.08 s ± 6.42 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [83]:
x = mvn_with_correlation(100, seed=0)
y = np.random.default_rng(seed=0).normal(size=100)
prop = equal_width_propositionalization(x) # full_propositionalization(x) #

In [98]:
search = SearchSpec(x, y, prop, 8)
res = run(search, weighted_support, weighted_support_bound_naive)
res.best.key, prop.str_from_conj(res.best.key), res.value

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)

In [100]:
res.nodes_created, res.edges_tested

(36523, 119843)

In [88]:
%timeit run(search, weighted_support, weighted_support_bound_naive)

343 ms ± 5.49 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [86]:
from numba import boolean

@jitclass
class WeightedSupportBound:

    pos: boolean[:]

    def __init__(self, y):
        self.pos = (y > 0)

    def compute(self, node, x, y):
        return y[node.support[self.pos[node.support]]].sum()

weighted_support_bound_pos_buffered = WeightedSupportBound(y)
res = run(search, weighted_support, weighted_support_bound_pos_buffered)
res.best.key, prop.str_from_conj(res.best.key), res.value

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)

 308 ms ± 5.92 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [90]:
%timeit run(search, weighted_support, weighted_support_bound_pos_buffered)

337 ms ± 2.27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [91]:
from numba import prange

@njit(fastmath=True)
def compute_weighted_support_bound_parallel(node, y):
    total = 0.0
    for i in prange(len(node.support)):
        idx = node.support[i]
        if y[idx] > 0.0:
            total += y[idx]
    return total

@jitclass
class ParallelWeightedSupportBound:

    def __init__(self):
        pass

    def compute(self, node, x, y):
        return compute_weighted_support_bound_parallel(node, y)
    

weighted_support_bound_parallel = ParallelWeightedSupportBound()
res = run(search, weighted_support, weighted_support_bound_parallel)
res.best.key, prop.str_from_conj(res.best.key), res.value

(array([ 8, 16, 23, 49, 59, 66]),
 'x1 >= -1.472 & x1 <= 1.740 & x2 >= 0.612 & x3 <= 1.730 & x4 >= -2.580 & x4 <= 0.538',
 20.397495605985817)

In [92]:
%timeit run(search, weighted_support, weighted_support_bound_parallel)

327 ms ± 2.43 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Attempt of Architecture with Node Polymorphism

- Different node types, e.g., `SimpleTreeSearchNode` and `PosNegTreeSearchNode`
- Function `run(spec, obj, bnd, make_root, refine)` receives both `make_root()` and `refine(node, spec)` functions.
- These can internally use the same node type (e.g. SimpleTreeSearchNode or PosNegTreeSearchNode), allowing full polymorphism by passing different root/refinement logic for each node type.

In [96]:
from numba import njit, int64, float64, types
from numba.experimental import jitclass
from numba.typed import List
import numpy as np

def make_heap_class(NodeType):
    TUPLE_TYPE = types.Tuple((float64, NodeType))

    heap_spec = [
        ('keys', types.ListType(float64)),
        ('nodes', types.ListType(NodeType))
    ]
    
    @jitclass(heap_spec)
    class Heap:
        def __init__(self):
            self.keys = List.empty_list(float64)
            self.nodes = List.empty_list(NodeType)

        def push(self, key, node):
            self.keys.append(key)
            self.nodes.append(node)
            i = len(self.keys) - 1
            self._sift_up(i)

        def pop(self):
            if len(self.keys) == 0:
                raise IndexError("pop from empty heap")
            top_key = self.keys[0]
            top_node = self.nodes[0]
            last_key = self.keys.pop()
            last_node = self.nodes.pop()
            if len(self.keys) == 0:
                return top_key, top_node
            self.keys[0] = last_key
            self.nodes[0] = last_node
            self._sift_down(0)
            return top_key, top_node

        def _sift_up(self, i):
            while i > 0:
                parent = (i - 1) // 2
                if self.keys[i] <= self.keys[parent]:
                    break
                self.keys[i], self.keys[parent] = self.keys[parent], self.keys[i]
                self.nodes[i], self.nodes[parent] = self.nodes[parent], self.nodes[i]
                i = parent

        def _sift_down(self, i):
            while True:
                left = 2 * i + 1
                right = 2 * i + 2
                largest = i
                if left < len(self.keys) and self.keys[left] > self.keys[largest]:
                    largest = left
                if right < len(self.keys) and self.keys[right] > self.keys[largest]:
                    largest = right
                if largest == i:
                    break
                self.keys[i], self.keys[largest] = self.keys[largest], self.keys[i]
                self.nodes[i], self.nodes[largest] = self.nodes[largest], self.nodes[i]
                i = largest

    return Heap

SimpleNodeHeap = make_heap_class(SimpleTreeSearchNodeType)
simple_node_heap = SimpleNodeHeap()
simple_root = make_simple_root(x, prop)
simple_node_heap.push(0, simple_root)
simple_node_heap.push(5, simple_root)
simple_node_heap.push(1, simple_root)
simple_node_heap.push(10, simple_root)
simple_node_heap.push(2, simple_root)
simple_node_heap.pop()

(10.0,
 <numba.experimental.jitclass.boxing.SimpleTreeSearchNode at 0x143235030>)

In [97]:
simple_node_heap.pop()

(5.0, <numba.experimental.jitclass.boxing.SimpleTreeSearchNode at 0x115251ba0>)

## Future Work

- Implement other priority keys, in particular arithmetic mean between value and bound. This requires storing bounds (or values) in nodes, because they are otherwise lost on retrieval for bound check.